# Task 3 — U-Net Semantic Segmentation on Stanford Background Dataset

This notebook runs all three loss-function experiments and logs results to TensorBoard.

**Experiments**
| # | Loss | Config key |
|---|------|------------|
| 1 | Cross-Entropy only | `loss_type='ce'` |
| 2 | Dice Loss only | `loss_type='dice'` |
| 3 | CE + Dice (combined) | `loss_type='combined'` |

**Before running**: download the Stanford Background Dataset and place it at `data/stanford_background/` (relative to the repo root), with sub-folders `images/` and `labels/`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
# Make sure the repo root is on the path so `src` is importable
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PyTorch 2.5.1+cu124
CUDA available: True
Using device: cuda


## 1. Imports

In [2]:
from src.config  import Config
from src.dataset import get_dataloaders
from src.model   import UNet
from src.losses  import get_loss_fn
from src.trainer import Trainer
from src.utils   import load_checkpoint, visualize_predictions

I0000 00:00:1778999797.941331 2817322 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 2. Shared hyperparameters

Edit the cell below to change any parameter. The `loss_type` field is overridden per-experiment.

In [3]:
BASE_CFG = dict(
    data_root      = "../iccv09Data",
    num_classes    = 8,
    input_size     = (256, 256),
    val_split      = 0.2,
    random_seed    = 42,

    # ── Training ──────────────────────────────────────────────────────────
    batch_size     = 32,
    num_epochs     = 50,
    learning_rate  = 5e-4,
    weight_decay   = 1e-4,
    num_workers    = 4,

    # ── Model ─────────────────────────────────────────────────────────────
    base_channels  = 64,

    # ── Optimizer / Scheduler ─────────────────────────────────────────────
    optimizer      = "adamw",
    scheduler      = "cosine",

    # ── Dice-specific (used when loss_type != 'ce') ────────────────────────
    dice_weight    = 0.3,
    dice_smooth    = 1e-6,

    # ── Logging ───────────────────────────────────────────────────────────
    log_dir        = "../runs",
    checkpoint_dir = "../checkpoints",
    save_every     = 10,
)

## 3. Helper: run one experiment

In [4]:
def run_experiment(loss_type: str, extra_cfg: dict = None) -> float:
    """
    Train U-Net with the given loss_type.
    Returns the best validation mIoU achieved.
    """
    cfg_kwargs = {**BASE_CFG, "loss_type": loss_type}
    if extra_cfg:
        cfg_kwargs.update(extra_cfg)

    cfg = Config(**cfg_kwargs)
    print(f"\n{'='*60}")
    print(f"Experiment: {cfg.experiment_name}")
    print(f"{'='*60}")

    train_loader, val_loader, class_counts = get_dataloaders(cfg)

    model   = UNet(num_classes=cfg.num_classes, base_channels=cfg.base_channels)
    loss_fn = get_loss_fn(cfg, class_counts=class_counts)

    print(f"Model parameters: {model.count_parameters():,}")

    trainer = Trainer(model, loss_fn, train_loader, val_loader, cfg, device)
    best_miou = trainer.train()
    return best_miou

## 4. Experiment 1 — Cross-Entropy Loss

In [5]:
miou_ce = run_experiment("ce")
print(f"\n[CE]  Best val mIoU = {miou_ce:.4f}")


Experiment: unet_ce_lr0.0005_bs32
Dataset: 574 train / 141 val samples
Model parameters: 31,385,288
TensorBoard logs → ../runs/unet_ce_lr0.0005_bs32
  Run:  tensorboard --logdir ../runs

Experiment : unet_ce_lr0.0005_bs32
Loss       : ce
Epochs     : 50  |  LR: 0.0005  |  BS: 32



unet_ce_lr0.0005_bs32: 100%|██████████| 50/50 [14:44<00:00, 17.68s/epoch, acc=0.8366, mIoU=0.6288, t=18.7s, tr_loss=0.2707, val_loss=0.4753]


Training complete. Best mIoU: 0.6326

[CE]  Best val mIoU = 0.6326


## 5. Experiment 2 — Dice Loss

In [6]:
miou_dice = run_experiment("dice")
print(f"\n[Dice]  Best val mIoU = {miou_dice:.4f}")


Experiment: unet_dice_lr0.0005_bs32
Dataset: 574 train / 141 val samples
Model parameters: 31,385,288
TensorBoard logs → ../runs/unet_dice_lr0.0005_bs32
  Run:  tensorboard --logdir ../runs

Experiment : unet_dice_lr0.0005_bs32
Loss       : dice
Epochs     : 50  |  LR: 0.0005  |  BS: 32



unet_dice_lr0.0005_bs32:   2%|▏         | 1/50 [00:20<16:43, 20.49s/epoch, acc=0.3163, mIoU=0.1459, t=18.8s, tr_loss=0.7774, val_loss=0.7796]

unet_dice_lr0.0005_bs32: 100%|██████████| 50/50 [16:10<00:00, 19.41s/epoch, acc=0.7512, mIoU=0.5354, t=18.6s, tr_loss=0.4123, val_loss=0.4108]


Training complete. Best mIoU: 0.5362

[Dice]  Best val mIoU = 0.5362


## 6. Experiment 3 — Combined Loss (CE + Dice)

In [7]:
miou_combined = run_experiment("combined")
print(f"\n[Combined]  Best val mIoU = {miou_combined:.4f}")


Experiment: unet_combined_lr0.0005_bs32
Dataset: 574 train / 141 val samples
Model parameters: 31,385,288
TensorBoard logs → ../runs/unet_combined_lr0.0005_bs32
  Run:  tensorboard --logdir ../runs

Experiment : unet_combined_lr0.0005_bs32
Loss       : combined
Epochs     : 50  |  LR: 0.0005  |  BS: 32



unet_combined_lr0.0005_bs32:   0%|          | 0/50 [00:18<?, ?epoch/s, acc=0.3917, mIoU=0.1682, t=18.5s, tr_loss=1.3087, val_loss=2.0090]

unet_combined_lr0.0005_bs32: 100%|██████████| 50/50 [16:28<00:00, 19.77s/epoch, acc=0.8402, mIoU=0.6428, t=17.2s, tr_loss=0.2500, val_loss=0.5032]


Training complete. Best mIoU: 0.6470

[Combined]  Best val mIoU = 0.6470


## 7. Results summary

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Loss Function": ["Cross-Entropy", "Dice", "CE + Dice"],
    "Best val mIoU": [miou_combined, miou_dice, miou_ce],
})
results["Best val mIoU"] = results["Best val mIoU"].map("{:.4f}".format)
print(results.to_string(index=False))

## 8. Launch TensorBoard

Run the cell below to open TensorBoard inline (works in Jupyter / VS Code).

In [9]:
# %load_ext tensorboard
# %tensorboard --logdir ../runs

## 9. Visualise predictions from the best model

Change `BEST_CKPT` to whichever checkpoint you want to inspect.

In [10]:
import matplotlib.pyplot as plt

BEST_LOSS_TYPE = "combined"   # change to "ce" or "dice" as needed

cfg = Config(**{**BASE_CFG, "loss_type": BEST_LOSS_TYPE})
_, val_loader, _ = get_dataloaders(cfg)

model = UNet(num_classes=cfg.num_classes, base_channels=cfg.base_channels).to(device)
ckpt_path = f"../checkpoints/{cfg.experiment_name}/best.pth"
load_checkpoint(model, ckpt_path, device=device)
model.eval()

images, labels = next(iter(val_loader))
with torch.no_grad():
    logits = model(images.to(device))

fig = visualize_predictions(
    images, labels, logits,
    class_names=cfg.CLASS_NAMES,
    save_path="../outputs/predictions.png",
    n_samples=4,
)
plt.show()

Dataset: 574 train / 141 val samples


/home/zhengxiang/CS60003-hw2/src/utils.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=device)


Loaded checkpoint from ../checkpoints/unet_combined_lr0.0005_bs32/best.pth  (epoch 43, mIoU 0.6326)


## 10. Ablation: vary learning rate

Uncomment and run to sweep learning rates with the combined loss.

In [11]:
# for lr in [1e-4, 5e-4, 1e-3, 3e-3]:
#     miou = run_experiment("combined", extra_cfg={"learning_rate": lr, "num_epochs": 30})
#     print(f"lr={lr}  →  best mIoU={miou:.4f}")